In [0]:
df= spark.read.table("samples.bakehouse.sales_transactions")
df2=df.where("paymentMethod == 'mastercard'")

display(df)

In [0]:
df2=df.where("paymentMethod == 'mastercard'")

display(df2)

In [0]:
%sql
select * from samples.bakehouse.sales_transactions where paymentMethod = 'mastercard'

In [0]:
from pyspark.sql.functions import expr, pow, round
#df2=df.withColumn("myCol",pow("quantity",2)/6)
df2=df.withColumn("myCol",round(pow("quantity",2)/6,2))
display(df2)

In [0]:
%sql
select *, round(pow(quantity,2)/6,2) from samples.bakehouse.sales_transactions

In [0]:
df2=df.select("product")\
.withColumn("lower_product",expr("lower(product)"))\
.withColumn("upper_product",expr("upper(product)"))\
.withColumn("trim_product",expr("trim(product)"))\
.withColumn("substr_product",expr("substr(product,0,4)"))\
.withColumn("split_product",expr("split(product,' ')"))

display(df2)

In [0]:
%sql
SELECT
    product,
    lower(product) AS lower_product,
    upper(product) AS upper_product,
    trim(product) AS trim_product,
    substr(product, 1, 3) AS substr_product,
    split(product, ' ') AS split_product
FROM samples.bakehouse.sales_transactions

In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace
df2=df.select("product").\
withColumn("regexp_extract_product", expr("regexp_extract(product, r'^\S+\s+(\S+)\s+\S+$', 1)")).\
withColumn("regexp_replace_product", expr("regexp_replace(product, r'^\S+\s+(\S+)\s+\S+$', 'AAAA')"))
display(df2)

In [0]:
%sql
SELECT product,
       regexp_extract(product, '(\\w+)', 1) AS extracted_word,
       regexp_replace(product, '\\s+', '_') AS replaced_spaces
FROM samples.bakehouse.sales_transactions;

In [0]:
from pyspark.sql.functions import current_date, current_timestamp, to_date, date_add, date_sub, lit

# 1. Obter data/hora atual e extrair apenas a data
df_dates = df.withColumn("today", current_date()) \
             .withColumn("now", current_timestamp()) \
             .withColumn("date_only", to_date("dateTime")) \
             .withColumn("plus_5_days", date_add(to_date("dateTime"), 5)) \
             .withColumn("minus_5_days", date_sub(to_date("dateTime"), 5))

# 2. Converter String formatada para o tipo Date especificando a máscara (yyyy-MM-dd)
df_custom_date = df.withColumn("parsed_date", to_date(lit("2025-10-14"), "yyyy-MM-dd"))

display(df_dates)

In [0]:
%sql
SELECT dateTime,
       current_date() AS today,
       current_timestamp() AS now,
       to_date(dateTime) AS date_only,
       date_add(to_date(dateTime), 5) AS plus_5_days,
       date_sub(to_date(dateTime), 5) AS minus_5_days,
       to_date('2025-10-14', 'yyyy-MM-dd') AS parsed_date
FROM samples.bakehouse.sales_transactions;

In [0]:
from pyspark.sql.functions import coalesce, expr, col

# 1. Coalesce: Retorna o primeiro valor NÃO NULO entre as colunas passadas
df_coalesce = df.select("transactionID", "customerID") \
                .withColumn("first_non_null", coalesce("transactionID", "customerID"))

# 2. Lógica condicional equivalente usando CASE WHEN (If-Else)
df_casewhen = df.withColumn("custom_logic", expr(
    "CASE WHEN transactionID IS NOT NULL THEN transactionID ELSE customerID END"
))

# 3. Remover linhas com valores nulos (na.drop)
df_dropped = df.na.drop(subset=["transactionID"])  
# Remove se transactionID for nulo
# df_dropped_all = df.na.drop("all")                 
# Remove se TODAS as colunas forem nulas

# 4. Preencher valores nulos com um valor padrão (na.fill)
df_filled = df.na.fill("valor_padrao")

display(df_coalesce)

In [0]:
%sql
-- 1 e 2. Uso de COALESCE e CASE WHEN em SQL
SELECT transactionID, customerID,
       coalesce(transactionID, customerID) AS first_non_null,
       CASE 
           WHEN transactionID IS NOT NULL THEN transactionID 
           ELSE customerID 
       END AS custom_logic
FROM samples.bakehouse.sales_transactions;

-- 3. Filtrar registros descartando nulos em SQL
SELECT * FROM samples.bakehouse.sales_transactions 
WHERE transactionID IS NOT NULL;